# Inference — attempt15 config, 3-seed soft-vote ensemble

Loads LoRA adapter weights from three independently trained seeds (same
hyperparameters as attempt15: r=8, simple prompt, 336px), runs loss-based
inference on the test set, and produces `submission.csv` via soft-vote.

**Setup**: mount your Drive and update `CKPT_DIRS` in cell 2 to point to
your three checkpoint folders.

In [ ]:
%pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow safetensors tqdm

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/DL/endterm')
    DATA_DIR = BASE_DIR / 'pixels-to-predictions'
else:
    BASE_DIR = Path('.')
    DATA_DIR = Path('pixels-to-predictions')

# ── Checkpoint directories (each must contain epoch_4/ and epoch_5/) ─────────
CKPT_DIRS = [
    BASE_DIR / 'checkpoints15',    # seed 1
    BASE_DIR / 'checkpoints15-1',  # seed 2
    BASE_DIR / 'checkpoints15-2',  # seed 3
]
ENSEMBLE_EPOCHS = [4, 5]  # late-epoch checkpoints only

print('Checkpoint dirs:')
for p in CKPT_DIRS:
    print(f'  {p}  exists={p.exists()}')

In [ ]:
import json, random
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
IMG_SIZE = 336

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    DTYPE  = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    DTYPE  = torch.float16
else:
    DEVICE = torch.device('cpu')
    DTYPE  = torch.float32

print(f'Device: {DEVICE} | dtype: {DTYPE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')
val_df  = pd.read_csv(DATA_DIR / 'val.csv')
for df in (test_df, val_df):
    df['choices'] = df['choices'].apply(json.loads)

mini_val_df = val_df.sample(200, random_state=SEED).reset_index(drop=True)

# auto-detect extra images/ nesting
IMG_DATA_DIR = DATA_DIR
if not (DATA_DIR / test_df.iloc[0]['image_path']).exists():
    if (DATA_DIR / 'images' / test_df.iloc[0]['image_path']).exists():
        IMG_DATA_DIR = DATA_DIR / 'images'

print(f'test={len(test_df)} | mini_val={len(mini_val_df)} | img_dir={IMG_DATA_DIR}')

In [ ]:
CHOICE_LETTERS = 'ABCDEFGHIJ'

def build_prompt(row, include_answer=False):
    parts = []
    for field in ('lecture', 'hint'):
        v = row.get(field, '')
        if pd.notna(v) and str(v).strip():
            parts.append(str(v).strip())
    ctx = '\n'.join(parts)
    choices_str = '\n'.join(
        f'  {CHOICE_LETTERS[i]}. {c}' for i, c in enumerate(row['choices'])
    )
    prompt = '<image>\n'
    if ctx:
        prompt += f'Context:\n{ctx}\n\n'
    prompt += f"Question: {row['question']}\nChoices:\n{choices_str}\nAnswer:"
    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"
    return prompt


class ScienceQADataset(Dataset):
    def __init__(self, df, img_dir, img_size=336):
        self.df       = df.reset_index(drop=True)
        self.img_dir  = img_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row['image_path']).convert('RGB').resize(
            (self.img_size, self.img_size), Image.BICUBIC
        )
        return {
            'image':   img,
            'text':    build_prompt(row, include_answer=False),
            'choices': row['choices'],
            'id':      row['id'],
            'answer':  int(row['answer']) if 'answer' in row.index else -1,
        }


test_ds     = ScienceQADataset(test_df,     IMG_DATA_DIR, IMG_SIZE)
mini_val_ds = ScienceQADataset(mini_val_df, IMG_DATA_DIR, IMG_SIZE)
print(f'test_ds={len(test_ds)} | mini_val_ds={len(mini_val_ds)}')

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType, set_peft_model_state_dict
import safetensors.torch as sf

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

load_kwargs = dict(dtype=DTYPE, low_cpu_mem_usage=True,
                   device_map='auto' if DEVICE.type == 'cuda' else None)
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, attn_implementation='flash_attention_2', **load_kwargs)
    print('FlashAttention-2 enabled')
except Exception as e:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
    print(f'Eager attention ({type(e).__name__})')

if DEVICE.type != 'cuda':
    model.to(DEVICE)

lora_cfg = LoraConfig(
    r=8, lora_alpha=16, use_dora=False,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj', 'out_proj'],
    lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.config.use_cache = False
model.eval()
model.print_trainable_parameters()


def load_ckpt(ckpt_path):
    weights = sf.load_file(str(ckpt_path / 'adapter_model.safetensors'), device='cpu')
    set_peft_model_state_dict(model, weights)
    del weights
    model.eval()

In [ ]:
@torch.inference_mode()
def predict_losses(sample):
    """Returns a list of per-choice cross-entropy losses (lower = preferred)."""
    img       = sample['image']
    base_text = sample['text']
    n         = len(sample['choices'])

    base_enc   = processor(text=[base_text], images=[img], return_tensors='pt')
    prompt_len = base_enc['input_ids'].shape[1]
    cached_pv  = base_enc['pixel_values'].to(DEVICE, dtype=DTYPE)
    cached_pam = base_enc.get('pixel_attention_mask')
    if cached_pam is not None:
        cached_pam = cached_pam.to(DEVICE)

    base_ids  = base_enc['input_ids']
    base_amsk = base_enc['attention_mask']
    losses    = []

    for i in range(n):
        ans_ids = processor.tokenizer.encode(
            f' {CHOICE_LETTERS[i]}', add_special_tokens=False, return_tensors='pt'
        )
        full_ids  = torch.cat([base_ids,  ans_ids],                  dim=1).to(DEVICE)
        full_amsk = torch.cat([base_amsk, torch.ones_like(ans_ids)], dim=1).to(DEVICE)

        labels = full_ids.clone()
        labels[:, :prompt_len] = -100
        labels[labels == processor.tokenizer.pad_token_id] = -100

        fwd = {'input_ids': full_ids, 'attention_mask': full_amsk,
               'pixel_values': cached_pv}
        if cached_pam is not None:
            fwd['pixel_attention_mask'] = cached_pam

        with torch.amp.autocast(device_type=DEVICE.type, dtype=DTYPE,
                                 enabled=DEVICE.type == 'cuda'):
            out = model(**fwd, labels=labels)

        n_toks = (labels != -100).sum().item()
        losses.append(out.loss.item() / max(n_toks, 1))
    return losses

In [ ]:
# ── Collect available checkpoint paths ────────────────────────────────────────
ckpt_paths = []
for ckpt_dir in CKPT_DIRS:
    for ep in ENSEMBLE_EPOCHS:
        p = ckpt_dir / f'epoch_{ep}'
        if (p / 'adapter_model.safetensors').exists():
            ckpt_paths.append(p)
            print(f'  found : {p}')
        else:
            print(f'  MISSING: {p}')

assert len(ckpt_paths) > 0, 'No checkpoints found — check CKPT_DIRS'
print(f'\nEnsemble size: {len(ckpt_paths)} checkpoints')

In [ ]:
from tqdm.auto import tqdm

# ── Soft-vote inference on test set ──────────────────────────────────────────
# For each checkpoint, score all test samples.
# Accumulate losses; argmin of the average gives the soft-vote prediction.

n_test      = len(test_ds)
accumulated = None  # list[np.ndarray], one per test sample

for ckpt_path in ckpt_paths:
    tag = f'{ckpt_path.parent.name}/{ckpt_path.name}'
    print(f'\nLoading {tag} ...')
    load_ckpt(ckpt_path)

    ckpt_losses = []
    for i in tqdm(range(n_test), desc=f'  {tag}', leave=False):
        ckpt_losses.append(np.array(predict_losses(test_ds[i])))

    if accumulated is None:
        accumulated = ckpt_losses
    else:
        for i in range(n_test):
            accumulated[i] = accumulated[i] + ckpt_losses[i]

n_ckpts     = len(ckpt_paths)
predictions = [int(np.argmin(accumulated[i] / n_ckpts)) for i in range(n_test)]
ids         = [test_ds[i]['id'] for i in range(n_test)]
print(f'\nSoft-vote complete ({n_ckpts} checkpoints).')

In [ ]:
# ── Mini-val sanity check (last loaded checkpoint) ───────────────────────────
correct = 0
for i in tqdm(range(len(mini_val_ds)), desc='val check'):
    if int(np.argmin(predict_losses(mini_val_ds[i]))) == mini_val_ds[i]['answer']:
        correct += 1
print(f'Mini-val accuracy (last ckpt): {correct}/{len(mini_val_ds)} = {correct/len(mini_val_ds):.4f}')

In [ ]:
# ── Save submission.csv ───────────────────────────────────────────────────────
sub = pd.DataFrame({'id': ids, 'answer': predictions})
sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv  ({len(sub)} rows)')
print(sub['answer'].value_counts().sort_index().rename('count').to_frame())

if IN_COLAB:
    out_path = BASE_DIR / 'submission.csv'
    sub.to_csv(out_path, index=False)
    print(f'Also saved to Drive: {out_path}')

sub.head(10)